# 03 — In Season Feature Engineering

This notebook creates the weekly performance features used by the in season projection layer.

It does not modify the original preseason notebooks or their outputs.

The purpose of this notebook is to convert completed regular season games into team level offensive and defensive performance features that can later be blended with the frozen preseason model.

For a selected target week, the notebook:

- Loads only completed games from weeks before the target week
- Builds team level scoring and margin summaries
- Separates offense and defense
- Adds simple opponent adjusted performance features
- Applies early season shrinkage to reduce overreaction to small samples
- Saves a clean feature table for the weekly team strength notebook

This notebook is intentionally conservative after Week 1. One game should inform the model, not redefine it.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the outputs created by:

`02_Weekly_Data_Update.ipynb`

The only value that normally changes week to week is `TARGET_WEEK`.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 10
PRIOR_WEEK = TARGET_WEEK - 1

completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

prior_strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

print("Using completed games:", completed_games_path)
print("Using entering ratings:", prior_strength_path)

Using completed games: ..\..\data\processed\weekly\week_10_completed_games.parquet
Using entering ratings: ..\..\data\processed\weekly\week_09_sequential_team_strength.parquet


# Load Completed Games

Only games completed before the target week should exist in this file.

That leakage rule was enforced in `02_Weekly_Data_Update.ipynb`.


In [3]:
completed_games = pd.read_parquet(
    completed_games_path
)

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 10: 136


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
...,...,...,...,...,...,...
131,9,2026-11-08,LV,17.0,SF,38.0
132,9,2026-11-08,NYG,19.0,PHI,11.0
133,9,2026-11-08,NYJ,12.0,KC,35.0
134,9,2026-11-08,TB,24.0,CHI,27.0


In [4]:
update_games = completed_games[
    completed_games["week"] == PRIOR_WEEK
].copy()

update_games = update_games.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Week {PRIOR_WEEK} games used to update "
    f"Week {TARGET_WEEK} ratings:",
    len(update_games)
)

display(
    update_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)

Week 9 games used to update Week 10 ratings: 15


,week,gameday,away_team,away_score,home_team,home_score
0,9,2026-11-05,JAX,33.0,BAL,10.0
1,9,2026-11-08,ARI,3.0,SEA,17.0
2,9,2026-11-08,CIN,34.0,ATL,17.0
3,9,2026-11-08,CLE,22.0,NO,23.0
4,9,2026-11-08,DAL,27.0,IND,23.0
5,9,2026-11-08,DEN,26.0,CAR,21.0
6,9,2026-11-08,DET,19.0,MIA,9.0
7,9,2026-11-08,GB,45.0,NE,27.0
8,9,2026-11-08,HOU,44.0,LAC,22.0
9,9,2026-11-08,LA,20.0,WAS,13.0


# Convert Prior Week Games to Team Game Rows

Only the most recently completed week is used for the new weekly update.

For Week 3:

- Week 2 team strength is the entering prior
- Week 2 game performance is the new evidence
- Week 1 is not counted again because it is already reflected in the Week 2 ratings

Each game becomes two rows:

- one from the home team's perspective
- one from the away team's perspective

In [5]:
home_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["home_team"],
        "opponent": update_games["away_team"],
        "is_home": 1,
        "points_for": update_games["home_score"],
        "points_against": update_games["away_score"]
    }
)

away_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["away_team"],
        "opponent": update_games["home_team"],
        "is_home": 0,
        "points_for": update_games["away_score"],
        "points_against": update_games["home_score"]
    }
)

team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

team_games["point_margin"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

team_games = team_games.sort_values(
    ["team", "game_id"]
).reset_index(drop=True)

print("Team-game rows:", len(team_games))

display(team_games)

Team-game rows: 30


,game_id,week,team,opponent,is_home,points_for,points_against,point_margin
0,2026_09_ARI_SEA,9,ARI,SEA,0,3.0,17.0,-14.0
1,2026_09_CIN_ATL,9,ATL,CIN,1,17.0,34.0,-17.0
2,2026_09_JAX_BAL,9,BAL,JAX,1,10.0,33.0,-23.0
3,2026_09_BUF_MIN,9,BUF,MIN,0,23.0,6.0,17.0
4,2026_09_DEN_CAR,9,CAR,DEN,1,21.0,26.0,-5.0
5,2026_09_TB_CHI,9,CHI,TB,1,27.0,24.0,3.0
6,2026_09_CIN_ATL,9,CIN,ATL,0,34.0,17.0,17.0
7,2026_09_CLE_NO,9,CLE,NO,0,22.0,23.0,-1.0
8,2026_09_DAL_IND,9,DAL,IND,0,27.0,23.0,4.0
9,2026_09_DEN_CAR,9,DEN,CAR,0,26.0,21.0,5.0


# Attach Entering Team and Opponent Strength

Each team's performance is evaluated relative to what the model believed before the game was played.

For Week 3 projections, Week 2 ratings represent the information available entering Week 2.

This prevents the opponent adjustment from using future information or the same results that are currently being evaluated.

In [6]:
prior_strength = pd.read_parquet(
    prior_strength_path
).copy()

required_prior_columns = [
    "team",
    "weekly_team_strength",
    "weekly_strength_rank"
]

missing_prior_columns = [
    col
    for col in required_prior_columns
    if col not in prior_strength.columns
]

if missing_prior_columns:
    raise KeyError(
        "Missing prior-week strength columns: "
        + ", ".join(missing_prior_columns)
    )

prior_strength = prior_strength[
    required_prior_columns
].rename(
    columns={
        "weekly_team_strength": "entering_team_strength",
        "weekly_strength_rank": "entering_strength_rank"
    }
)

print(
    f"Week {PRIOR_WEEK} entering ratings loaded:",
    len(prior_strength)
)

display(
    prior_strength.sort_values(
        "entering_strength_rank"
    )
)

Week 9 entering ratings loaded: 32


,team,entering_team_strength,entering_strength_rank
0,BUF,6.296234,1
1,JAX,4.547996,2
2,LA,4.376166,3
3,DEN,4.105679,4
4,SF,3.931716,5
5,SEA,3.749348,6
6,CHI,3.577705,7
7,DET,3.225150,8
8,MIN,2.861256,9
9,PIT,2.844367,10


In [7]:
team_games = team_games.merge(
    prior_strength[
        [
            "team",
            "entering_team_strength",
            "entering_strength_rank"
        ]
    ],
    on="team",
    how="left",
    validate="many_to_one"
)

opponent_strength = prior_strength[
    [
        "team",
        "entering_team_strength",
        "entering_strength_rank"
    ]
].rename(
    columns={
        "team": "opponent",
        "entering_team_strength": "opponent_entering_strength",
        "entering_strength_rank": "opponent_entering_rank"
    }
)

team_games = team_games.merge(
    opponent_strength,
    on="opponent",
    how="left",
    validate="many_to_one"
)

display(
    team_games[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "opponent",
            "opponent_entering_rank",
            "opponent_entering_strength",
            "points_for",
            "points_against",
            "point_margin"
        ]
    ].sort_values(
        "entering_strength_rank"
    )
)

,team,entering_strength_rank,entering_team_strength,opponent,opponent_entering_rank,opponent_entering_strength,points_for,points_against,point_margin
3,BUF,1,6.296234,MIN,9,2.861256,23.0,6.0,17.0
14,JAX,2,4.547996,BAL,17,-0.354664,33.0,10.0,23.0
16,LA,3,4.376166,WAS,19,-1.602244,20.0,13.0,7.0
9,DEN,4,4.105679,CAR,27,-3.816203,26.0,21.0,5.0
27,SF,5,3.931716,LV,24,-3.272264,38.0,17.0,21.0
26,SEA,6,3.749348,ARI,28,-4.337349,17.0,3.0,14.0
5,CHI,7,3.577705,TB,15,1.097898,27.0,24.0,3.0
10,DET,8,3.225150,MIA,30,-4.601078,19.0,9.0,10.0
20,MIN,9,2.861256,BUF,1,6.296234,6.0,23.0,-17.0
21,NE,11,2.832965,GB,13,2.104000,27.0,45.0,-18.0


# Expected Margin and Opponent-Adjusted Performance

The weekly update is based on how a team performed relative to expectation.

For each team game:

`performance residual = actual margin - expected margin`

Expected margin uses the difference between the two teams' entering strength ratings plus home field advantage.

This means beating a strong opponent is more informative than producing the same result against a weak opponent.

In [8]:
HOME_FIELD_POINTS = 1.5

team_games["expected_margin"] = (
    team_games["entering_team_strength"]
    - team_games["opponent_entering_strength"]
    + np.where(
        team_games["is_home"] == 1,
        HOME_FIELD_POINTS,
        -HOME_FIELD_POINTS
    )
)

team_games["performance_residual"] = (
    team_games["point_margin"]
    - team_games["expected_margin"]
)

display(
    team_games[
        [
            "team",
            "opponent",
            "entering_team_strength",
            "opponent_entering_strength",
            "point_margin",
            "expected_margin",
            "performance_residual"
        ]
    ].sort_values(
        "performance_residual",
        ascending=False
    )
)

,team,opponent,entering_team_strength,opponent_entering_strength,point_margin,expected_margin,performance_residual
11,GB,NE,2.104000,2.832965,18.0,-2.228965,20.228965
14,JAX,BAL,4.547996,-0.354664,23.0,3.402660,19.597340
12,HOU,LAC,1.010378,-3.424708,22.0,2.935086,19.064914
3,BUF,MIN,6.296234,2.861256,17.0,1.934977,15.065023
6,CIN,ATL,1.808836,-1.644763,17.0,1.953600,15.046400
15,KC,NYJ,2.693362,-5.190536,23.0,9.383898,13.616102
27,SF,LV,3.931716,-3.272264,21.0,8.703980,12.296020
23,NYG,PHI,-2.376744,-3.331484,8.0,-0.545260,8.545260
26,SEA,ARI,3.749348,-4.337349,14.0,9.586697,4.413303
10,DET,MIA,3.225150,-4.601078,10.0,6.326227,3.673773


# Control Extreme Single-Game Results

NFL game margins contain substantial single-game noise.

A blowout should affect team strength, but one extreme result should not completely redefine the team's rating.

Performance residuals are therefore capped before the weekly learning rate is applied.

In [9]:
RESIDUAL_SCALE = 14.0
MAX_WEEKLY_UPDATE = 1.5

team_games["weekly_strength_update"] = (
    MAX_WEEKLY_UPDATE
    * np.tanh(
        team_games["performance_residual"]
        / RESIDUAL_SCALE
    )
)

display(
    team_games[
        [
            "team",
            "opponent",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,opponent,point_margin,expected_margin,performance_residual,weekly_strength_update
11,GB,NE,18.0,-2.228965,20.228965,1.342028
14,JAX,BAL,23.0,3.402660,19.597340,1.327966
12,HOU,LAC,22.0,2.935086,19.064914,1.315208
3,BUF,MIN,17.0,1.934977,15.065023,1.187607
6,CIN,ATL,17.0,1.953600,15.046400,1.186862
15,KC,NYJ,23.0,9.383898,13.616102,1.124753
27,SF,LV,21.0,8.703980,12.296020,1.058340
23,NYG,PHI,8.0,-0.545260,8.545260,0.816587
26,SEA,ARI,14.0,9.586697,4.413303,0.457789
10,DET,MIA,10.0,6.326227,3.673773,0.384826


# Offensive and Defensive Weekly Signals

Overall team strength uses the opponent adjusted margin residual.

We also preserve separate offensive and defensive signals because the weekly game projection model needs scoring information.

These are weekly updates, not cumulative season averages.

Opponent entering strength is used conservatively as context until separate entering offensive and defensive strength ratings are maintained.

In [10]:
league_points_per_team_game = (
    team_games["points_for"].mean()
)

OPPONENT_CONTEXT_RATE = 0.25
OFFENSE_DEFENSE_UPDATE_RATE = 0.15

team_games["raw_offense_performance"] = (
    team_games["points_for"]
    - league_points_per_team_game
)

team_games["raw_defense_performance"] = (
    league_points_per_team_game
    - team_games["points_against"]
)

team_games["adjusted_offense_performance"] = (
    team_games["raw_offense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["adjusted_defense_performance"] = (
    team_games["raw_defense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["weekly_offense_adjustment"] = (
    team_games["adjusted_offense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

team_games["weekly_defense_adjustment"] = (
    team_games["adjusted_defense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

display(
    team_games[
        [
            "team",
            "opponent",
            "points_for",
            "points_against",
            "opponent_entering_strength",
            "raw_offense_performance",
            "adjusted_offense_performance",
            "weekly_offense_adjustment",
            "raw_defense_performance",
            "adjusted_defense_performance",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "team"
    )
)

,team,opponent,points_for,points_against,opponent_entering_strength,raw_offense_performance,adjusted_offense_performance,weekly_offense_adjustment,raw_defense_performance,adjusted_defense_performance,weekly_defense_adjustment
0,ARI,SEA,3.0,17.0,3.749348,-19.233333,-18.295996,-2.744399,5.233333,6.170670,0.925601
1,ATL,CIN,17.0,34.0,1.808836,-5.233333,-4.781124,-0.717169,-11.766667,-11.314458,-1.697169
2,BAL,JAX,10.0,33.0,4.547996,-12.233333,-11.096334,-1.664450,-10.766667,-9.629668,-1.444450
3,BUF,MIN,23.0,6.0,2.861256,0.766667,1.481981,0.222297,16.233333,16.948647,2.542297
4,CAR,DEN,21.0,26.0,4.105679,-1.233333,-0.206914,-0.031037,-3.766667,-2.740247,-0.411037
5,CHI,TB,27.0,24.0,1.097898,4.766667,5.041141,0.756171,-1.766667,-1.492192,-0.223829
6,CIN,ATL,34.0,17.0,-1.644763,11.766667,11.355476,1.703321,5.233333,4.822143,0.723321
7,CLE,NO,22.0,23.0,-2.535573,-0.233333,-0.867227,-0.130084,-0.766667,-1.400560,-0.210084
8,DAL,IND,27.0,23.0,-2.754096,4.766667,4.078143,0.611721,-0.766667,-1.455191,-0.218279
9,DEN,CAR,26.0,21.0,-3.816203,3.766667,2.812616,0.421892,1.233333,0.279283,0.041892


# Build Week-Level Feature Table

The final feature table contains only the new evidence that should move the prior week's ratings.

For Week 3:

`Week 3 strength = Week 2 strength + Week 2 performance update`

The previous week's rating therefore acts as the prior, and older games are not counted again.

In [11]:
team_features = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_in_update=("game_id", "count"),

        entering_team_strength=(
            "entering_team_strength",
            "first"
        ),

        entering_strength_rank=(
            "entering_strength_rank",
            "first"
        ),

        average_opponent_strength=(
            "opponent_entering_strength",
            "mean"
        ),

        average_opponent_rank=(
            "opponent_entering_rank",
            "mean"
        ),

        actual_margin=(
            "point_margin",
            "mean"
        ),

        expected_margin=(
            "expected_margin",
            "mean"
        ),

        performance_residual=(
            "performance_residual",
            "mean"
        ),

        weekly_strength_update=(
            "weekly_strength_update",
            "mean"
        ),

        weekly_offense_adjustment=(
            "weekly_offense_adjustment",
            "mean"
        ),

        weekly_defense_adjustment=(
            "weekly_defense_adjustment",
            "mean"
        )
    )
)

display(
    team_features[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "average_opponent_rank",
            "average_opponent_strength",
            "actual_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,entering_strength_rank,entering_team_strength,average_opponent_rank,average_opponent_strength,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
11,GB,13,2.104000,11.0,2.832965,18.0,-2.228965,20.228965,1.342028,3.521236,-0.608764
14,JAX,2,4.547996,17.0,-0.354664,23.0,3.402660,19.597340,1.327966,1.601700,1.821700
12,HOU,16,1.010378,26.0,-3.424708,22.0,2.935086,19.064914,1.315208,3.136573,-0.093427
3,BUF,1,6.296234,9.0,2.861256,17.0,1.934977,15.065023,1.187607,0.222297,2.542297
6,CIN,14,1.808836,20.0,-1.644763,17.0,1.953600,15.046400,1.186862,1.703321,0.723321
15,KC,12,2.693362,31.0,-5.190536,23.0,9.383898,13.616102,1.124753,1.720355,1.340355
27,SF,5,3.931716,24.0,-3.272264,21.0,8.703980,12.296020,1.058340,2.242290,0.662290
23,NYG,21,-2.376744,25.0,-3.331484,8.0,-0.545260,8.545260,0.816587,-0.609931,1.560069
26,SEA,6,3.749348,28.0,-4.337349,14.0,9.586697,4.413303,0.457789,-0.947651,2.722349
10,DET,8,3.225150,30.0,-4.601078,10.0,6.326227,3.673773,0.384826,-0.657540,1.812460


# Sanity Checks

Before saving, verify that:

- all 32 teams are represented
- every team has an entering Week 2 rating
- every opponent has an entering Week 2 rating
- no Week 3 game results were used
- weekly strength updates remain within the intended cap

In [12]:
print("Teams in feature table:", len(team_features))
print(
    "Unique teams:",
    team_features["team"].nunique()
)

print(
    "Missing entering team strengths:",
    team_features[
        "entering_team_strength"
    ].isna().sum()
)

print(
    "Missing opponent strengths:",
    team_features[
        "average_opponent_strength"
    ].isna().sum()
)

print(
    "Largest positive weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].max(),
        3
    )
)

print(
    "Largest negative weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].min(),
        3
    )
)

if (
    completed_games["week"] >= TARGET_WEEK
).any():
    raise ValueError(
        "DATA LEAKAGE: target-week results "
        "were found in completed games."
    )

if len(team_features) != 32:
    print()
    print(
        "WARNING: Feature table does not "
        "contain all 32 teams."
    )

Teams in feature table: 30
Unique teams: 30
Missing entering team strengths: 0
Missing opponent strengths: 0
Largest positive weekly update: 1.342
Largest negative weekly update: -1.342



# Save In Season Feature Table

This output will be consumed by:

`04_Weekly_Team_Strength.ipynb`

The file contains only the new in season evidence. It does not overwrite or replace the preseason team strength table.


In [13]:
output_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

team_features = team_features.sort_values(
    "team"
).reset_index(drop=True)

team_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: ..\..\data\processed\weekly\week_10_inseason_features.parquet


# Interpretation

The weekly model is now sequential.

For Week 3:

1. Week 2 team strength is treated as the entering prior.
2. Only Week 2 game results provide new evidence.
3. Performance is evaluated relative to the opponent's entering strength.
4. Extreme single game residuals are capped.
5. A conservative learning rate determines the weekly rating movement.
6. The resulting update will be added to the Week 2 rating in `04_Weekly_Team_Strength.ipynb`.

This prevents older games from being counted repeatedly and makes strength of opponent part of every weekly update.

The frozen preseason model remains unchanged.